## 경로 설정 (레포 공통)

이 셀은 레포 어디에서 노트북을 열어도 `data/` 폴더를 자동으로 찾아 **상대경로 기반**으로 경로를 지정한다.
- `data/raw` 원자료 → `data/interim` 중간 자료 → `data/processed` 최종본, 그림·표는 `outputs/`에 저장한다.
- 노트북은 레포 안 어느 위치에서 실행해도 되지만, 레포 폴더 구조(`data/`, `src/`, `outputs/`)를 바꾸지 않아야 한다.

In [1]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "data" / "raw").exists())
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
FIG_DIR = ROOT / "outputs" / "figures"
TAB_DIR = ROOT / "outputs" / "tables"
for _d in (INTERIM, PROCESSED, FIG_DIR, TAB_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print("프로젝트 루트 폴더:", ROOT.name)

프로젝트 루트 폴더: BAF-26-2-finance_2


# 안정성 원자료 검증

## 목적

이 노트북은 `안정성_전체통합_1994_2025.csv`가 **후속 분석에 사용 가능한 원자료인지 확인**하기 위한 최소한의 품질 검증을 수행한다.

원자료 단계에서는 업종별 장기 추세나 세부 분포를 깊게 해석하지 않는다.  
본격적인 EDA는 이후 `재무_원자재_병합패널.csv` 등 **최종 분석용 병합패널**에서 수행한다.

### 이번 노트북에서 확인하는 내용
1. 데이터 기본 구조
2. 패널 구조의 정상 여부
3. 결측·중복·무한대·0값·음수값
4. 눈에 띄는 극단 관측치
5. 주요 안정성 지표의 내부 산식 일관성
6. 원자료 사용 가능 여부에 대한 최종 판단

## 1. 데이터 불러오기

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# financial_project/notebooks/ 아래에서 실행하는 것을 기준으로 함
file_path = INTERIM / "ISTANS 안정성" / "안정성_전체통합_1994_2025.csv"

df = pd.read_csv(file_path)

print(f"파일 존재 여부: {file_path.exists()}")
print(f"데이터 크기: {df.shape[0]:,}행 × {df.shape[1]}열")

파일 존재 여부: True
데이터 크기: 1,280행 × 21열


## 2. 기본 구조 확인

행·열 수, 변수명, 자료형, 결측 여부를 확인한다.  
이 단계의 목적은 **파일이 예상한 형태로 구성되어 있는지** 빠르게 점검하는 것이다.

In [3]:
basic_info = pd.DataFrame({
    "변수명": df.columns,
    "자료형": df.dtypes.astype(str).values,
    "결측개수": df.isna().sum().values,
    "결측비율(%)": (df.isna().mean() * 100).round(2).values,
    "고유값개수": df.nunique(dropna=True).values
})

display(basic_info)

print(f"업종 수: {df['업종'].nunique()}")
print(f"연도 수: {df['연도'].nunique()}")
print(f"기간: {df['연도'].min()} ~ {df['연도'].max()}")

,변수명,자료형,결측개수,결측비율(%),고유값개수
0,업종코드,str,0,0.0,40
1,업종,str,0,0.0,40
2,연도,int64,0,0.0,32
3,자산총계,float64,0,0.0,1247
4,유형자산,float64,0,0.0,1194
5,유동자산,float64,0,0.0,1216
6,재고자산,float64,0,0.0,1074
7,자본총계,float64,0,0.0,1214
8,이자비용,float64,0,0.0,600
9,부채총계,float64,0,0.0,1241


업종 수: 40
연도 수: 32
기간: 1994 ~ 2025


### 확인 결과

- 총 40개 업종과 32개 연도가 존재한다.
- 1994~2025년 기간의 자료다.
- 모든 변수에서 결측치는 확인되지 않는다.
- 업종코드·업종·연도와 재무 수치형 변수의 자료형도 분석에 사용 가능한 형태다.

## 3. 패널 구조 검증

원자료가 **40개 업종 × 32개 연도**의 완전한 패널인지 확인한다.

확인 항목:
- 업종코드와 업종명이 1:1로 대응하는가
- 동일 업종-연도 조합이 중복되는가
- 누락된 업종-연도 조합이 있는가
- 모든 연도에 40개 업종이 존재하는가

In [4]:
code_name_problem = (
    df.groupby("업종코드")["업종"].nunique()
      .reset_index(name="업종명_개수")
      .query("업종명_개수 != 1")
)

name_code_problem = (
    df.groupby("업종")["업종코드"].nunique()
      .reset_index(name="업종코드_개수")
      .query("업종코드_개수 != 1")
)

duplicate_panel = df[
    df.duplicated(subset=["업종코드", "연도"], keep=False)
].copy()

all_industries = df[["업종코드", "업종"]].drop_duplicates()
all_years = pd.DataFrame({"연도": sorted(df["연도"].unique())})

expected_panel = (
    all_industries.assign(key=1)
    .merge(all_years.assign(key=1), on="key")
    .drop(columns="key")
)

actual_panel = df[["업종코드", "업종", "연도"]].drop_duplicates()

missing_panel = (
    expected_panel
    .merge(
        actual_panel,
        on=["업종코드", "업종", "연도"],
        how="left",
        indicator=True
    )
    .query('_merge == "left_only"')
    .drop(columns="_merge")
)

year_industry_count = (
    df.groupby("연도")["업종코드"]
      .nunique()
      .reset_index(name="관측업종수")
)

is_balanced = (
    len(code_name_problem) == 0
    and len(name_code_problem) == 0
    and len(duplicate_panel) == 0
    and len(missing_panel) == 0
    and year_industry_count["관측업종수"].eq(40).all()
)

panel_result = pd.DataFrame({
    "확인 항목": [
        "업종코드-업종명 불일치",
        "업종-연도 중복",
        "누락 업종-연도 조합",
        "균형패널 여부"
    ],
    "결과": [
        len(code_name_problem) + len(name_code_problem),
        len(duplicate_panel),
        len(missing_panel),
        "예" if is_balanced else "아니오"
    ]
})

display(panel_result)

,확인 항목,결과
0,업종코드-업종명 불일치,0
1,업종-연도 중복,0
2,누락 업종-연도 조합,0
3,균형패널 여부,예


### 확인 결과

- 업종코드와 업종명은 1:1로 대응한다.
- 동일 업종-연도 중복은 없다.
- 누락된 업종-연도 조합도 없다.
- 모든 연도에서 40개 업종이 존재한다.

따라서 본 자료는 **40개 업종을 32년간 관측한 완전 균형패널**이다.

## 4. 기본 품질 점검

결측치 외에도 무한대, 0값, 음수값이 존재하는지 확인한다.

이 단계에서는 0이나 음수를 곧바로 오류로 판단하지 않는다.  
재무자료에서는 실제로 차입이 없거나 자기자본이 음수인 경우가 존재할 수 있기 때문이다.

In [5]:
value_cols = df.columns[3:]

quality_rows = []

for col in value_cols:
    s = df[col]
    quality_rows.append({
        "변수명": col,
        "결측": s.isna().sum(),
        "무한대": np.isinf(s).sum(),
        "0값": (s == 0).sum(),
        "0값비율(%)": round((s == 0).mean() * 100, 2),
        "음수값": (s < 0).sum(),
        "음수비율(%)": round((s < 0).mean() * 100, 2)
    })

quality_check = pd.DataFrame(quality_rows)
display(quality_check)

,변수명,결측,무한대,0값,0값비율(%),음수값,음수비율(%)
0,자산총계,0,0,0,0.00,0,0.00
1,유형자산,0,0,1,0.08,0,0.00
2,유동자산,0,0,0,0.00,0,0.00
3,재고자산,0,0,0,0.00,0,0.00
4,자본총계,0,0,0,0.00,6,0.47
5,이자비용,0,0,2,0.16,0,0.00
6,부채총계,0,0,0,0.00,0,0.00
7,유동부채,0,0,0,0.00,0,0.00
8,비유동부채,0,0,1,0.08,0,0.00
9,단기차입,0,0,1,0.08,0,0.00


### 확인 결과

- 결측치와 무한대는 없다.
- `단기사채` 등 일부 차입 관련 변수에는 0값이 많이 존재한다. 이는 실제로 해당 조달수단을 사용하지 않은 **구조적 0**일 가능성이 있으므로 임의로 제거하지 않는다.
- `자본총계` 및 일부 비율 변수에서는 6개의 음수값이 확인된다. 해당 관측치만 추가 확인한다.

## 5. 주요 이상 관측치 확인

In [6]:
check_cols = [
    "자본총계",
    "자본비율",
    "부채비율",
    "유동부채비율",
    "비유동부채비율",
    "차입의존도"
]

negative_rows = df[
    (df[check_cols] < 0).any(axis=1)
][
    [
        "업종코드", "업종", "연도",
        "자산총계", "부채총계", "자본총계",
        "자본비율", "부채비율",
        "유동부채비율", "비유동부채비율",
        "차입의존도"
    ]
].sort_values(["업종", "연도"])

display(negative_rows)

,업종코드,업종,연도,자산총계,부채총계,자본총계,자본비율,부채비율,유동부채비율,비유동부채비율,차입의존도
34,I3106,가전,1998,2550.0,2556.0,-6.0,-0.2,-43356.6,-35584.2,-7769.7,-25381.6
35,I3106,가전,1999,2966.0,3063.0,-97.0,-3.3,-3144.8,-2563.0,-581.8,-1638.6
107,I3209,기타 수송장비,1999,209.0,214.0,-5.0,-2.2,-4624.4,-1766.3,-2858.1,-2793.0
410,I3209,기타 수송장비,2000,148.0,209.0,-60.0,-40.8,-345.0,-137.2,-207.8,-200.6
411,I3209,기타 수송장비,2001,172.0,222.0,-50.0,-29.1,-444.1,-226.6,-217.5,-215.5
177,I3312,조선,1997,10107.0,10251.0,-144.0,-1.4,-7118.5,-5579.4,-1539.0,-4633.6


### 해석

음수값이 나타난 관측치는 총 6건으로, 가전·기타 수송장비·조선의 과거 연도에 집중되어 있다.

이 관측치들은 `자본총계 < 0`이며, 같은 행에서 자본을 기준으로 계산되는 안정성 비율도 동시에 음수로 나타난다.

특히 `자산총계 ≈ 부채총계 + 자본총계` 관계가 유지되므로, 현재 단계에서는 단순 입력오류라기보다 **음의 자기자본에서 발생한 구조적 극단값**으로 본다.

따라서 원자료 단계에서는 해당 관측치를 삭제하지 않는다.  
최종 분석용 병합패널에 실제로 포함되는지 여부는 이후 병합패널 검증 단계에서 다시 확인한다.

## 6. 주요 안정성 지표 산식 검증

공식 원자료이므로 모든 항목을 재계산할 필요는 없지만,
최종 분석에 사용할 가능성이 높은 안정성 비율이 원자료 금액과 대체로 일관적인지 확인한다.

특히 `차입의존도`는 다른 지표와 달리 산식 정의가 불명확할 가능성이 있어 별도로 비교한다.

In [7]:
formula_check = df.copy()

formula_check["자본비율_calc"] = (
    formula_check["자본총계"] / formula_check["자산총계"] * 100
)

formula_check["부채비율_calc"] = (
    formula_check["부채총계"] / formula_check["자본총계"] * 100
)

formula_check["유동부채비율_calc"] = (
    formula_check["유동부채"] / formula_check["자본총계"] * 100
)

formula_check["비유동부채비율_calc"] = (
    formula_check["비유동부채"] / formula_check["자본총계"] * 100
)

formula_check["차입금합계"] = (
    formula_check["단기차입"]
    + formula_check["장기차입"]
    + formula_check["단기사채"]
    + formula_check["장기사채"]
)

formula_check["차입의존도_자산기준"] = (
    formula_check["차입금합계"] / formula_check["자산총계"] * 100
)

formula_check["차입의존도_자본기준"] = (
    formula_check["차입금합계"] / formula_check["자본총계"] * 100
)

result = pd.DataFrame({
    "비율": [
        "자본비율",
        "부채비율",
        "유동부채비율",
        "비유동부채비율",
        "차입의존도 - 자산 기준",
        "차입의존도 - 자본 기준"
    ],
    "중앙값 절대오차": [
        (formula_check["자본비율"] - formula_check["자본비율_calc"]).abs().median(),
        (formula_check["부채비율"] - formula_check["부채비율_calc"]).abs().median(),
        (formula_check["유동부채비율"] - formula_check["유동부채비율_calc"]).abs().median(),
        (formula_check["비유동부채비율"] - formula_check["비유동부채비율_calc"]).abs().median(),
        (formula_check["차입의존도"] - formula_check["차입의존도_자산기준"]).abs().median(),
        (formula_check["차입의존도"] - formula_check["차입의존도_자본기준"]).abs().median()
    ]
})

display(result.round(4))

,비율,중앙값 절대오차
0,자본비율,0.0263
1,부채비율,0.0297
2,유동부채비율,0.0290
3,비유동부채비율,0.0296
4,차입의존도 - 자산 기준,22.9322
5,차입의존도 - 자본 기준,0.0319


### 해석

- `자본비율`, `부채비율`, `유동부채비율`, `비유동부채비율`은 재계산값과의 중앙값 절대오차가 약 0.03%p 수준으로 매우 작다.
- 따라서 주요 안정성 지표는 원자료 금액과 내부적으로 일관적인 것으로 판단한다.
- `차입의존도`는 자산총계보다 자본총계를 분모로 계산한 값과 훨씬 가깝게 나타난다.

다만 이 결과는 **데이터 내부의 산식 일치 여부**만 보여준다.  
`차입의존도`의 공식 정의는 원 통계 메타데이터를 확인한 후 최종 분석에 사용할지 결정한다.

## 7. 원자료 검증 결론

### 최종 판단

안정성 원자료는 다음 조건을 충족한다.

- 40개 업종 × 32년의 완전 균형패널
- 업종코드와 업종명의 1:1 대응
- 업종-연도 중복 없음
- 결측치 없음
- 무한대값 없음
- 주요 안정성 비율의 내부 산식 일관성 확인

일부 과거 연도에서 음의 자기자본에 따른 극단적인 비율값이 존재하지만,
기본 회계 관계가 유지되므로 단순 입력오류로 판단하지 않고 원자료에 그대로 보존한다.

따라서 **안정성 원자료는 후속 분석에 사용 가능한 것으로 판단한다.**

단, `차입의존도`는 공식 지표 정의를 추가 확인한 뒤 최종 분석 사용 여부를 결정한다.

---

### 이후 분석 원칙

이 원자료에서는 여기까지 검증하고 종료한다.

다음과 같은 심화 EDA는 **최종 분석용 병합패널**에서 수행한다.

- 변수별 분포
- 극단값의 최종 분석표본 포함 여부
- 업종별·연도별 패턴
- 상관관계
- within / between variation
- 원자재 노출도와 재무지표 관계
- 평행추세 및 DiD 식별 점검